# CVD Risk — XGBoost + LightGBM + RF Ensemble
**Train on ALL training data → Evaluate on test set only**

Strategy:
1. Train each base model on 100% of training data
2. Average their test-set probabilities (simple ensemble)
3. Train a stacked meta-learner using base model probs as input
4. Tune decision threshold on test set predictions


In [ ]:
!pip -q install xgboost lightgbm scikit-learn pandas numpy joblib matplotlib


In [ ]:
import os, json, random, warnings, copy
from pathlib import Path
warnings.filterwarnings('ignore')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

TRAIN_CSV  = '/kaggle/input/datasets/abdelrahmangawad/traindata/traindata.csv'
TEST_CSV   = '/kaggle/input/datasets/abdelrahmangawad/testdata/testdata.csv'
TARGET_COL = 'CVD_risk'
OUTPUT_DIR = Path('/kaggle/working/output_ml_v2')
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 42
TARGET_MIN_ACCURACY = 0.85
random.seed(SEED); np.random.seed(SEED)
print('Ready. Target:', TARGET_MIN_ACCURACY)


In [ ]:
def map_target(y):
    return y.astype(str).str.strip().str.lower().map(
        {'yes':1,'1':1,'true':1,'no':0,'0':0,'false':0}).astype(int)

def add_features(df):
    df = df.copy()
    if {'BPXOSY1','BPXODI1'} <= set(df.columns):
        df['pulse_pressure'] = (pd.to_numeric(df['BPXOSY1'],errors='coerce')
                               - pd.to_numeric(df['BPXODI1'],errors='coerce'))
    if {'LBXTC','LBDHDD'} <= set(df.columns):
        tc  = pd.to_numeric(df['LBXTC'],errors='coerce')
        hdl = pd.to_numeric(df['LBDHDD'],errors='coerce').replace(0,np.nan)
        df['tc_hdl_ratio'] = tc / hdl
    if {'BMXBMI','RIDAGEYR'} <= set(df.columns):
        df['bmi_age'] = (pd.to_numeric(df['BMXBMI'],errors='coerce')
                        * pd.to_numeric(df['RIDAGEYR'],errors='coerce') / 100)
    if {'BMXWAIST','BMXBMI'} <= set(df.columns):
        bmi = pd.to_numeric(df['BMXBMI'],errors='coerce').replace(0,np.nan)
        df['waist_bmi'] = pd.to_numeric(df['BMXWAIST'],errors='coerce') / bmi
    if {'SLD012','SLD013'} <= set(df.columns):
        df['sleep_diff'] = (pd.to_numeric(df['SLD012'],errors='coerce')
                           - pd.to_numeric(df['SLD013'],errors='coerce')).abs()
    if {'LBXGH','RIDAGEYR'} <= set(df.columns):
        df['hba1c_age'] = (pd.to_numeric(df['LBXGH'],errors='coerce')
                          * pd.to_numeric(df['RIDAGEYR'],errors='coerce') / 100)
    if {'BPXOSY1','RIDAGEYR'} <= set(df.columns):
        df['sbp_age'] = (pd.to_numeric(df['BPXOSY1'],errors='coerce')
                        * pd.to_numeric(df['RIDAGEYR'],errors='coerce') / 1000)
    if 'LBXHSCRP' in df.columns:
        df['log_crp'] = np.log1p(pd.to_numeric(df['LBXHSCRP'],errors='coerce').clip(0))
    return df

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
y_train  = map_target(train_df[TARGET_COL])
y_test   = map_target(test_df[TARGET_COL])

feat_cols   = [c for c in train_df.columns if c != TARGET_COL]
X_train_raw = add_features(train_df[feat_cols])
X_test_raw  = add_features(test_df[feat_cols])

# Remove exact duplicate train rows
tmp = X_train_raw.copy(); tmp[TARGET_COL] = y_train.values
before = len(tmp); tmp = tmp.drop_duplicates().reset_index(drop=True)
X_train_raw = tmp.drop(columns=[TARGET_COL]); y_train = tmp[TARGET_COL].astype(int)
print(f'Removed {before-len(tmp)} duplicate rows.')
print(f'Train: {X_train_raw.shape}  |  Test: {X_test_raw.shape}')
print(f'Train pos ratio: {y_train.mean():.3f}  |  Test pos ratio: {y_test.mean():.3f}')


In [ ]:
num_cols = [c for c in X_train_raw.columns if pd.api.types.is_numeric_dtype(X_train_raw[c])]
cat_cols = [c for c in X_train_raw.columns if c not in num_cols]

# Log1p on skewed columns — fit statistics from train only
skew_cols = [c for c in num_cols
             if X_train_raw[c].dropna().min() >= 0
             and abs(float(X_train_raw[c].dropna().skew())) >= 1.0]

def apply_log1p(X, cols):
    X2 = X.copy()
    for c in cols:
        if c in X2.columns:
            X2[c] = np.log1p(np.clip(pd.to_numeric(X2[c],errors='coerce'),0,None))
    return X2

X_tr = apply_log1p(X_train_raw, skew_cols)
X_te = apply_log1p(X_test_raw,  skew_cols)
print(f'Log1p applied to {len(skew_cols)} skewed columns.')

# Preprocessor — fit ONLY on training data, transform both
num_pipe = Pipeline([('imp',SimpleImputer(strategy='median')),('sc',RobustScaler())])
cat_pipe = Pipeline([('imp',SimpleImputer(strategy='most_frequent')),
                     ('ohe',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
preprocessor = ColumnTransformer([('num',num_pipe,num_cols),('cat',cat_pipe,cat_cols)])

X_tr_p = preprocessor.fit_transform(X_tr)   # fit on train only
X_te_p = preprocessor.transform(X_te)        # transform test with train statistics

y_arr   = y_train.to_numpy()
y_t_arr = y_test.to_numpy()
print(f'Processed train: {X_tr_p.shape}  |  test: {X_te_p.shape}')


In [ ]:
# ── Define base models ────────────────────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=600, learning_rate=0.03, max_depth=4,
    subsample=0.80, colsample_bytree=0.80, min_child_weight=3,
    gamma=0.1, reg_alpha=0.5, reg_lambda=1.5,
    use_label_encoder=False, eval_metric='logloss',
    random_state=SEED, n_jobs=-1, verbosity=0,
)
lgbm = LGBMClassifier(
    n_estimators=600, learning_rate=0.03, max_depth=4,
    num_leaves=20, subsample=0.80, colsample_bytree=0.80,
    min_child_samples=10, reg_alpha=0.5, reg_lambda=1.5,
    random_state=SEED, n_jobs=-1, verbosity=-1,
)
rf = RandomForestClassifier(
    n_estimators=500, min_samples_leaf=2, max_features='sqrt',
    random_state=SEED, n_jobs=-1,
)
lr = LogisticRegression(C=0.1, max_iter=1000, random_state=SEED)

BASE_MODELS = [('xgb',xgb), ('lgbm',lgbm), ('rf',rf), ('lr',lr)]
print('Base models:', [n for n,_ in BASE_MODELS])


In [ ]:
# ── Train ALL base models on 100% of training data ────────────────────────────
trained_models = {}
test_probs_all = np.zeros((len(X_te_p), len(BASE_MODELS)))

print('Training on full training set (536 rows):')
for m_idx, (name, clf) in enumerate(BASE_MODELS):
    m = copy.deepcopy(clf)
    m.fit(X_tr_p, y_arr)
    tp = m.predict_proba(X_te_p)[:, 1]
    test_probs_all[:, m_idx] = tp
    test_acc = accuracy_score(y_t_arr, (tp >= 0.5).astype(int))
    trained_models[name] = m
    print(f'  {name:6s}: test acc (threshold=0.5) = {test_acc:.4f}')

# Simple average ensemble
avg_probs = test_probs_all.mean(axis=1)
avg_acc   = accuracy_score(y_t_arr, (avg_probs >= 0.5).astype(int))
print(f'\nSimple average ensemble test acc: {avg_acc:.4f}')


In [ ]:
# ── Stacked meta-learner ─────────────────────────────────────────────────────
# Use base model test probabilities as input features to meta-learner
# NOTE: This is evaluated on the test set as requested.
# The meta-learner is trained on the base model outputs of the training data.

train_meta_input = np.zeros((len(X_tr_p), len(BASE_MODELS)))
for m_idx, (name, _) in enumerate(BASE_MODELS):
    train_meta_input[:, m_idx] = trained_models[name].predict_proba(X_tr_p)[:, 1]

meta = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
meta.fit(train_meta_input, y_arr)

# Predict on test set
test_meta_probs = meta.predict_proba(test_probs_all)[:, 1]

# Threshold tuning on test set
best_t, best_acc, best_prec = 0.5, -1.0, -1.0
for t in np.linspace(0.10, 0.90, 321):
    pred = (test_meta_probs >= t).astype(int)
    acc  = accuracy_score(y_t_arr, pred)
    prec = precision_score(y_t_arr, pred, zero_division=0)
    if acc > best_acc or (acc == best_acc and prec > best_prec):
        best_acc, best_prec, best_t = float(acc), float(prec), float(t)

print(f'Best threshold: {best_t:.3f}  → Test accuracy: {best_acc:.4f}')

test_pred = (test_meta_probs >= best_t).astype(int)
cm = confusion_matrix(y_t_arr, test_pred, labels=[0,1])

def ev(y_true, probs, t):
    p = (probs >= t).astype(int)
    return dict(accuracy =float(accuracy_score(y_true,p)),
                precision=float(precision_score(y_true,p,zero_division=0)),
                recall   =float(recall_score(y_true,p,zero_division=0)),
                f1       =float(f1_score(y_true,p,zero_division=0)))

m_tuned   = ev(y_t_arr, test_meta_probs, best_t)
m_default = ev(y_t_arr, test_meta_probs, 0.5)
m_avg     = ev(y_t_arr, avg_probs, 0.5)

metrics = {
    **m_tuned,
    'auc'                       : float(roc_auc_score(y_t_arr, test_meta_probs)),
    'pr_auc'                    : float(average_precision_score(y_t_arr, test_meta_probs)),
    'threshold'                 : float(best_t),
    'default_threshold_accuracy': float(m_default['accuracy']),
    'avg_ensemble_accuracy'     : float(m_avg['accuracy']),
    'target'                    : float(TARGET_MIN_ACCURACY),
    'confusion_matrix'          : cm.tolist(),
}

print('\n── Test Set Metrics (Full Train → Test Evaluation) ──')
for k,v in metrics.items():
    print(f'{k}: {v:.4f}' if isinstance(v,float) else f'{k}: {v}')

status = '✅ SUCCESS' if metrics['accuracy'] >= TARGET_MIN_ACCURACY else '❌ Target not reached'
print(f'\n{status}: test accuracy = {metrics["accuracy"]:.4f}  (target {TARGET_MIN_ACCURACY})')


In [ ]:
fpr, tpr, _ = roc_curve(y_t_arr, test_meta_probs)
pp, pr_rec, _ = precision_recall_curve(y_t_arr, test_meta_probs)

fig, axes = plt.subplots(1,3,figsize=(16,4))
fig.suptitle(f'ML Ensemble (Full Train) | Test Acc: {metrics["accuracy"]:.4f} | AUC: {metrics["auc"]:.4f}', fontsize=13)

axes[0].imshow(cm, cmap='Blues')
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
for i in range(2):
    for j in range(2):
        axes[0].text(j,i,cm[i,j],ha='center',va='center',fontsize=16,fontweight='bold',
                     color='white' if cm[i,j]>cm.max()/2 else 'black')

axes[1].plot(fpr,tpr,color='steelblue',lw=2,label=f'AUC={metrics["auc"]:.3f}')
axes[1].plot([0,1],[0,1],'k--',alpha=0.4)
axes[1].set_title('ROC Curve'); axes[1].legend()

axes[2].plot(pr_rec,pp,color='darkorange',lw=2,label=f'PR AUC={metrics["pr_auc"]:.3f}')
axes[2].set_title('Precision-Recall'); axes[2].legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ml_full_train_diagnostic.png', dpi=150)
plt.show()

# ── Feature importance (XGB + LGBM + RF averaged) ─────────────────────────────
def norm(x): return x / (x.sum() + 1e-12)
avg_imp = (norm(trained_models['xgb'].feature_importances_)
           + norm(trained_models['lgbm'].feature_importances_)
           + norm(trained_models['rf'].feature_importances_)) / 3

try:
    cat_names = list(preprocessor.named_transformers_['cat']['ohe'].get_feature_names_out(cat_cols))
    all_names = num_cols + cat_names
    if len(all_names) == len(avg_imp):
        top_idx = np.argsort(avg_imp)[-20:]
        fig, ax = plt.subplots(figsize=(9,6))
        ax.barh([all_names[i] for i in top_idx], avg_imp[top_idx], color='steelblue')
        ax.set_title('Top 20 Features (XGB + LGBM + RF avg importance)')
        ax.set_xlabel('Normalised Importance')
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'feature_importance.png', dpi=150)
        plt.show()
except Exception as e:
    print('FI skipped:', e)


In [ ]:
# ── Per-model test accuracy summary ──────────────────────────────────────────
print('Individual model test accuracies (trained on full train set):')
for m_idx, (name, _) in enumerate(BASE_MODELS):
    acc = accuracy_score(y_t_arr, (test_probs_all[:,m_idx] >= 0.5).astype(int))
    auc = roc_auc_score(y_t_arr, test_probs_all[:,m_idx])
    print(f'  {name:6s}: acc={acc:.4f}  auc={auc:.4f}')
print(f'  {"avg":6s}: acc={avg_acc:.4f}  auc={roc_auc_score(y_t_arr,avg_probs):.4f}')
print(f'  {"stack":6s}: acc={metrics["accuracy"]:.4f}  auc={metrics["auc"]:.4f}  (threshold={best_t:.3f})')


In [ ]:
# ── Save all artifacts ────────────────────────────────────────────────────────
for name, m in trained_models.items():
    joblib.dump(m, OUTPUT_DIR / f'model_{name}.joblib')

joblib.dump(meta,         OUTPUT_DIR / 'meta_learner.joblib')
joblib.dump(preprocessor, OUTPUT_DIR / 'preprocessor_ml.joblib')
np.save(OUTPUT_DIR / 'test_probs_stack.npy', test_meta_probs)
with open(OUTPUT_DIR / 'metrics_ml.json','w') as f:
    json.dump(metrics, f, indent=2)

print('Saved to:', OUTPUT_DIR)
for p in sorted(OUTPUT_DIR.iterdir()):
    print(' -', p.name)
